<a href="https://colab.research.google.com/github/Aniebiet1/Agentic_AI/blob/main/cafx_pred_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Exchange Data
       ↓
#Candle Dataset Builder
       ↓
#Feature Engineering (LIVE = TRAIN identical)
       ↓
#Labeling Engine (TP hit before SL)
       ↓
#Train/Test Split (time-based)
       ↓
#Model (LightGBM / XGBoost)
       ↓
#Backtest Simulator
       ↓
# Live Prediction Engine

# 1. INSTALL DEPENDENCIES

In [ ]:
!pip install ccxt ta lightgbm scikit-learn pandas numpy

# 2. DATA PIPELINE (FETCH + CLEAN)

In [ ]:
import ccxt
import pandas as pd

exchange = ccxt.okx({"enableRateLimit": True})

SYMBOL = "BTC/USDT"
TIMEFRAME = "1m"


def fetch_ohlcv(limit=2000):
    data = exchange.fetch_ohlcv(SYMBOL, TIMEFRAME, limit=limit)

    df = pd.DataFrame(data, columns=[
        "timestamp", "open", "high", "low", "close", "volume"
    ])

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)

    return df

# 3. FEATURE ENGINEERING

In [ ]:
def add_features(df):
    df = df.copy()

    df["ret_1"] = df["close"].pct_change()
    df["ret_5"] = df["close"].pct_change(5)

    df["body_ratio"] = abs(df["close"] - df["open"]) / (df["high"] - df["low"] + 1e-9)

    df["ema_9"] = df["close"].ewm(span=9).mean()
    df["ema_21"] = df["close"].ewm(span=21).mean()
    df["ema_slope"] = df["ema_9"] - df["ema_21"]


    # RSI
    df["rsi"] = RSIIndicator(df["close"], window=14).rsi()

    # MACD
    macd = MACD(df["close"])
    df["macd"] = macd.macd()
    df["macd_signal"] = macd.macd_signal()

    # volume
    df["volume_z"] = (
        (df["volume"] - df["volume"].rolling(20).mean()) /
        (df["volume"].rolling(20).std() + 1e-9)
    )

    # ATR proxy
    df["atr"] = (
        df["high"].rolling(14).max() -
        df["low"].rolling(14).min()
    )

    # Hour
    df["hour"] = df["timestamp"].dt.hour

    # fill missing
    df = df.ffill().bfill()

    return df

# 4. LABELING ENGINE (MOST IMPORTANT PART)
This is your core ML logic.

Goal:

“Did TP hit before SL within next N candles?”

In [ ]:
import numpy as np
from ta.momentum import RSIIndicator
from ta.trend import MACD

def create_labels(df, horizon=60, tp_mult=1.0, sl_mult=1.0):

    labels = []

    for i in range(len(df) - horizon):

        entry = df.iloc[i]["close"]
        atr = df.iloc[i]["atr"]

        tp = entry + tp_mult * atr
        sl = entry - sl_mult * atr

        future = df.iloc[i+1:i+horizon]

        label = None

        for j in range(len(future)):

            high = future.iloc[j]["high"]
            low = future.iloc[j]["low"]

            if high >= tp:
                label = 1
                break

            if low <= sl:
                label = 0
                break

        if label is None:
            label = 0  # treat unclear as loss

        labels.append(label)

    df = df.iloc[:len(labels)].copy()
    df["label"] = labels

    return df

# 5. BUILD TRAINING DATASET

In [ ]:
df = fetch_ohlcv(10000)
df = add_features(df)
df = create_labels(df)
print(df["label"].value_counts(normalize=True))

min_class = df["label"].value_counts().min()

df_balanced = pd.concat([
    df[df["label"] == 0].sample(min_class, random_state=42),
    df[df["label"] == 1].sample(min_class, random_state=42)
])

df = df_balanced.sample(frac=1, random_state=42)

label
0    0.779167
1    0.220833
Name: proportion, dtype: float64


# 6. FEATURES + TARGET SPLIT

In [ ]:
FEATURES = [
    "ret_1","ret_5",
    "body_ratio",
    "ema_slope",
    "rsi",
    "macd",
    "macd_signal",
    "volume_z",
    "hour"
]

X = df[FEATURES]
y = df["label"]

# 7. TRAIN / TEST SPLIT (TIME SERIES SAFE)

In [ ]:
split = int(len(df) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 8. TRAIN MODEL (LIGHTGBM BEST OPTION) and save model


In [ ]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    class_weight="balanced"
)

model.fit(X_train, y_train)

import pickle

artifact = {
    "model": model,
    "features": FEATURES
}

with open("trading_model.pkl", "wb") as f:
    pickle.dump(artifact, f)

print("MODEL SAVED")

[LightGBM] [Info] Number of positive: 41, number of negative: 43
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000216 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 233
[LightGBM] [Info] Number of data points in the train set: 84, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

# COMPLETE PROFIT VALIDATION CODE

In [ ]:
capital = 1000

risk_per_trade = 0.01

wins = 0
losses = 0

total_profit = 0

trade_results = []

test_df = df.iloc[split:].copy()

test_df["prediction"] = model.predict(X_test)

trade_log = []

# ============================================
# LOOP THROUGH TEST SET
# ============================================

for i in range(len(test_df)):

    row = test_df.iloc[i]

    pred = row["prediction"]

    actual = row["label"]

    # ========================================
    # ONLY TAKE LONG TRADES
    # ========================================

    if pred == 1:

        risk_amount = capital * risk_per_trade

        # RR = 1.5 : 1
        reward_amount = risk_amount * 1.5

        # ==============================
        # WIN
        # ==============================

        if actual == 1:

            capital += reward_amount

            total_profit += reward_amount

            wins += 1

            trade_results.append(reward_amount)
            trade_log.append({ "result": "WIN","profit": reward_amount})

        # ==============================
        # LOSS
        # ==============================

        else:

            capital -= risk_amount

            total_profit -= risk_amount

            losses += 1

            trade_results.append(-risk_amount)
            trade_log.append({
                  "result": "LOSS",
                    "profit": -risk_amount
                      })

## PERFORMANCE METRICS

In [ ]:
total_trades = wins + losses

win_rate = wins / total_trades if total_trades > 0 else 0

profit_factor = (
    sum([x for x in trade_results if x > 0]) /
    abs(sum([x for x in trade_results if x < 0]))
) if losses > 0 else 0

expectancy = np.mean(trade_results)

print("\n========== BACKTEST RESULTS ==========")

print("Final Capital:", round(capital, 2))

print("Total Profit:", round(total_profit, 2))

print("Wins:", wins)

print("Losses:", losses)

print("Win Rate:", round(win_rate * 100, 2), "%")

print("Profit Factor:", round(profit_factor, 2))

print("Expectancy:", round(expectancy, 2))

pd.DataFrame(trade_log).to_csv(
    "backtest_results.csv",
    index=False
)

print("BACKTEST SAVED")


========== BACKTEST RESULTS ==========
Final Capital: 1114.81
Total Profit: 114.81
Wins: 10
Losses: 4
Win Rate: 71.43 %
Profit Factor: 3.7
Expectancy: 8.2
BACKTEST SAVED


# 9. EVALUATION

In [ ]:
from sklearn.metrics import classification_report

preds = model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.75      0.60      0.67        10
           1       0.71      0.83      0.77        12

    accuracy                           0.73        22
   macro avg       0.73      0.72      0.72        22
weighted avg       0.73      0.73      0.72        22



# 10. SIMPLE BACKTEST LOGIC

In [ ]:
def backtest(df, model, features):
    df = df.copy()

    df["prob"] = model.predict_proba(df[features])[:,1]

    df["signal"] = "HOLD"
    df.loc[df["prob"] > 0.6, "signal"] = "LONG"
    df.loc[df["prob"] < 0.4, "signal"] = "SHORT"

    return df

# 11. LIVE INFERENCE RULE

In [ ]:
def predict_latest(df, model):
    latest = df.iloc[-1:][FEATURES]

    prob = model.predict_proba(latest)[0][1]


    if prob > 0.65:
        return "LONG", prob
    elif prob < 0.35:
        return "SHORT", prob
    else:
        return "NO_TRADE", prob

In [ ]:
import ccxt
import pandas as pd

exchange = ccxt.okx()

candles = exchange.fetch_ohlcv(
    "BTC/USDT",
    timeframe="1m",
    limit=5
)

df = pd.DataFrame(
    candles,
    columns=[
        "timestamp",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
)

print(df)

       timestamp     open     high      low    close    volume
0  1779092880000  77002.3  77021.0  77002.2  77008.9  1.713210
1  1779092940000  77008.9  77026.9  76978.3  76978.4  2.547841
2  1779093000000  76978.3  76998.7  76951.0  76951.0  2.164507
3  1779093060000  76950.9  76961.1  76936.7  76948.3  2.022847
4  1779093120000  76948.3  76959.3  76917.4  76957.0  4.537893


# Live Simulation


In [ ]:
import time
import pandas as pd
import numpy as np

# =========================================================
# ACTIVE TRADE
# =========================================================

active_trade = None

# =========================================================
# LOG STORAGE
# =========================================================

live_logs = []

# =========================================================
# LAST CANDLE TRACKER
# =========================================================

last_timestamp = None

# =========================================================
# WAIT FOR NEXT CLOSED CANDLE
# =========================================================

def wait_for_next_candle():

    now = pd.Timestamp.utcnow()

    wait = 60 - now.second + 2

    print(f"\nWaiting {wait} sec for candle close...\n")

    time.sleep(wait)

# =========================================================
# LIVE LOOP
# =========================================================

while True:

    try:

        # =================================================
        # WAIT FOR CLOSED CANDLE
        # =================================================

        wait_for_next_candle()

        # =================================================
        # FETCH DATA
        # =================================================

        df = fetch_ohlcv(200)

        # =================================================
        # FEATURE ENGINEERING
        # =================================================

        df = add_features(df)

        # =================================================
        # SAFETY CHECK
        # =================================================

        if df is None or len(df) < 50:

            print("Not enough data")

            continue

        # =================================================
        # USE LAST CLOSED CANDLE
        # =================================================

        latest = df.iloc[-2:-1]

        # =================================================
        # DUPLICATE CANDLE PROTECTION
        # =================================================

        current_timestamp = str(
            latest["timestamp"].values[0]
        )

        if current_timestamp == last_timestamp:

            print("Duplicate candle detected")

            continue

        last_timestamp = current_timestamp

        # =================================================
        # PREPARE FEATURES
        # =================================================

        X_live = latest[FEATURES]

        # =================================================
        # PREDICT
        # =================================================

        prob = model.predict_proba(X_live)[0][1]

        long_prob = prob

        short_prob = 1 - prob

        # =================================================
        # MARKET INFO
        # =================================================

        price = float(
            latest["close"].values[0]
        )

        atr = float(
            latest["atr"].values[0]
        )

        high = float(
            latest["high"].values[0]
        )

        low = float(
            latest["low"].values[0]
        )

        candle_time = str(
            latest["timestamp"].values[0]
        )

        print("=" * 60)

        print(f"TIME: {candle_time}")

        print(f"PRICE: {price:.2f}")

        print(f"LONG PROB: {long_prob:.4f}")

        print(f"SHORT PROB: {short_prob:.4f}")

        print("=" * 60)

        # =================================================
        # MANAGE ACTIVE TRADE
        # =================================================

        if active_trade is not None:

            active_trade["candles_open"] += 1

            print("\nACTIVE TRADE:")

            print(active_trade)

            # =============================================
            # LONG MANAGEMENT
            # =============================================

            if active_trade["type"] == "LONG":

                if high >= active_trade["tp"]:

                    print("\nWIN LONG")

                    profit = (
                        active_trade["tp"]
                        - active_trade["entry"]
                    )

                    live_logs.append({

                        "time": candle_time,

                        "event": "CLOSE",

                        "result": "WIN",

                        "trade_type": "LONG",

                        "entry": active_trade["entry"],

                        "exit": active_trade["tp"],

                        "tp": active_trade["tp"],

                        "sl": active_trade["sl"],

                        "profit": profit,

                        "probability":
                        active_trade["probability"],

                        "candles_held":
                        active_trade["candles_open"]
                    })

                    active_trade = None

                elif low <= active_trade["sl"]:

                    print("\nLOSS LONG")

                    profit = (
                        active_trade["sl"]
                        - active_trade["entry"]
                    )

                    live_logs.append({

                        "time": candle_time,

                        "event": "CLOSE",

                        "result": "LOSS",

                        "trade_type": "LONG",

                        "entry": active_trade["entry"],

                        "exit": active_trade["sl"],

                        "tp": active_trade["tp"],

                        "sl": active_trade["sl"],

                        "profit": profit,

                        "probability":
                        active_trade["probability"],

                        "candles_held":
                        active_trade["candles_open"]
                    })

                    active_trade = None

            # =============================================
            # SHORT MANAGEMENT
            # =============================================

            elif active_trade["type"] == "SHORT":

                if low <= active_trade["tp"]:

                    print("\nWIN SHORT")

                    profit = (
                        active_trade["entry"]
                        - active_trade["tp"]
                    )

                    live_logs.append({

                        "time": candle_time,

                        "event": "CLOSE",

                        "result": "WIN",

                        "trade_type": "SHORT",

                        "entry": active_trade["entry"],

                        "exit": active_trade["tp"],

                        "tp": active_trade["tp"],

                        "sl": active_trade["sl"],

                        "profit": profit,

                        "probability":
                        active_trade["probability"],

                        "candles_held":
                        active_trade["candles_open"]
                    })

                    active_trade = None

                elif high >= active_trade["sl"]:

                    print("\nLOSS SHORT")

                    profit = (
                        active_trade["entry"]
                        - active_trade["sl"]
                    )

                    live_logs.append({

                        "time": candle_time,

                        "event": "CLOSE",

                        "result": "LOSS",

                        "trade_type": "SHORT",

                        "entry": active_trade["entry"],

                        "exit": active_trade["sl"],

                        "tp": active_trade["tp"],

                        "sl": active_trade["sl"],

                        "profit": profit,

                        "probability":
                        active_trade["probability"],

                        "candles_held":
                        active_trade["candles_open"]
                    })

                    active_trade = None

            # =============================================
            # TIME EXIT
            # =============================================

            if active_trade is not None:

                if active_trade["candles_open"] >= 60:

                    print("\nTIME EXIT")

                    live_logs.append({

                        "time": candle_time,

                        "event": "TIME_EXIT",

                        "trade_type":
                        active_trade["type"],

                        "entry":
                        active_trade["entry"],

                        "tp":
                        active_trade["tp"],

                        "sl":
                        active_trade["sl"],

                        "probability":
                        active_trade["probability"],

                        "candles_held":
                        active_trade["candles_open"]
                    })

                    active_trade = None

        # =================================================
        # OPEN NEW TRADE
        # =================================================

        if active_trade is None:

            # =============================================
            # LONG SIGNAL
            # =============================================

            if long_prob > 0.60:

                active_trade = {

                    "type": "LONG",

                    "entry": price,

                    "tp": price + 1.5 * atr,

                    "sl": price - 1.0 * atr,

                    "probability": long_prob,

                    "opened_at": candle_time,

                    "candles_open": 0
                }

                print("\nLONG OPENED")

                print(active_trade)

                live_logs.append({

                    "time": candle_time,

                    "event": "OPEN",

                    "trade_type": "LONG",

                    "entry": active_trade["entry"],

                    "tp": active_trade["tp"],

                    "sl": active_trade["sl"],

                    "probability": long_prob
                })

            # =============================================
            # SHORT SIGNAL
            # =============================================

            elif short_prob > 0.60:

                active_trade = {

                    "type": "SHORT",

                    "entry": price,

                    "tp": price - 1.5 * atr,

                    "sl": price + 1.0 * atr,

                    "probability": short_prob,

                    "opened_at": candle_time,

                    "candles_open": 0
                }

                print("\nSHORT OPENED")

                print(active_trade)

                live_logs.append({

                    "time": candle_time,

                    "event": "OPEN",

                    "trade_type": "SHORT",

                    "entry": active_trade["entry"],

                    "tp": active_trade["tp"],

                    "sl": active_trade["sl"],

                    "probability": short_prob
                })

        # =================================================
        # SAVE LOGS
        # =================================================

        pd.DataFrame(
            live_logs
        ).to_csv(
            "live_simulation.csv",
            index=False
        )

    except Exception as e:

        print("\nERROR:", e)

        time.sleep(30)


Waiting 12 sec for candle close...

TIME: 2026-05-18T08:32:00.000000000
PRICE: 76957.00
LONG PROB: 0.9749
SHORT PROB: 0.0251

LONG OPENED
{'type': 'LONG', 'entry': 76957.0, 'tp': 77168.65000000001, 'sl': 76815.9, 'probability': np.float64(0.9749216833781357), 'opened_at': '2026-05-18T08:32:00.000000000', 'candles_open': 0}

Waiting 60 sec for candle close...

TIME: 2026-05-18T08:33:00.000000000
PRICE: 76978.10
LONG PROB: 0.7827
SHORT PROB: 0.2173

ACTIVE TRADE:
{'type': 'LONG', 'entry': 76957.0, 'tp': 77168.65000000001, 'sl': 76815.9, 'probability': np.float64(0.9749216833781357), 'opened_at': '2026-05-18T08:32:00.000000000', 'candles_open': 1}

Waiting 59 sec for candle close...

TIME: 2026-05-18T08:34:00.000000000
PRICE: 76987.90
LONG PROB: 0.9758
SHORT PROB: 0.0242

ACTIVE TRADE:
{'type': 'LONG', 'entry': 76957.0, 'tp': 77168.65000000001, 'sl': 76815.9, 'probability': np.float64(0.9749216833781357), 'opened_at': '2026-05-18T08:32:00.000000000', 'candles_open': 2}

Waiting 60 sec fo